In [25]:

import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import networkx as nx
import random
import heapq
import collections

### All the simulation uses hours and kilometers

In [26]:
LAMBDA_T = [314.2, 162.4, 138.6, 148.8, 273.2, 1118.8, 2773.8, 4036.2, 4237.4, 3277.0, 2843.0, 2876.4, 3143.0, 3277.8, 3546.2, 4335.0, 4945.4, 4525.8, 2847.8, 1828.0, 1378.4, 1271.2, 1171.2, 767.6 ]

In [27]:
#Sampling arrival times of cars to network
def lambdat(t : np.array):
    lambdat = []
    for time in t:
        lambdat.append(LAMBDA_T[int(np.floor(time))])
    return lambdat

def arrival_times(lam): #Taken from lecture notes
    max_T = 24
    arrival_times = collections.deque()
    exp_dist = stats.expon(scale = 1/lam)
    t = exp_dist.rvs()
    while t < max_T:
        arrival_times.append(t)
        t += exp_dist.rvs()
    
    return np.asarray(arrival_times)

In [28]:
Graph = nx.read_gml('./data/networkAssignment.gml')
JUNCTIONS = list(Graph.nodes)

In [29]:
for e in Graph.edges:
    Graph.edges[e]['accident'] = False
    Graph.edges[e]['accident_duration'] = 0

In [30]:
## ADDED THIS FOR DEBUGGING PURPOSE ONLY ##

selected_edges = random.sample(list(Graph.edges), 3)

for u, v in selected_edges:
    # Set the accident attributes for each selected edge
    Graph[u][v]['accident'] = True
    Graph[u][v]['accident_duration'] = .1



In [31]:
Graph.edges[('1410566272', '8432860337')]

{'name': 'Knooppunt Gouwe->Knooppunt Terbregseplein',
 'highway': 'motorway_link',
 'length': 11557.0,
 'lanes': 2,
 'accident': True,
 'accident_duration': 0.1}

In [32]:
class FES:
    def __init__(self):
        self.events = []

    def add(self, event):
        heapq.heappush(self.events, event)
    
    def next(self):
        return heapq.heappop(self.events)
    
    def isEmpty(self):
        return len(self.events) == 0
    
    def __repr__(self):
        string = ''
        sorted_events = sorted(self.events)
        for event in sorted_events:
            string += f'{event}\n'
        return string

In [33]:
class Event:
    TYPE = ['New car', 'Car departure', 'Accident']
    def __init__(self, typ:int, time, car = None, road = None):
        #types:
            #0 : Arrival of car to the network
            #1 : Car leaves current road and goes on to the next
            #2 : Accident in road
            #5 : Solve Accident
        self.type = typ
        self.time = time
        self.road = road

        if typ == 0:
            car = Car(time_entrance = time)
    
        self.car = car
        
    def __str__(self):
        if self.type == 0:
            return f'{self.TYPE[self.type]} from {self.car.origin} to {self.car.destination} at {self.time}'
        if self.type == 1:
            return f'{self.TYPE[self.type]} of {self.car} at {self.time}h'
        if self.type == 2:
            return f'{self.TYPE[self.type]} at {self.road} at {self.time}h'

    def __lt__(self, other):
        return self.time < other.time

In [75]:
class Car:
    VELOCITIES = [100, 80]
    VELOCITIES_P = [0.9, 0.1]
    def __init__(self, time_entrance, origin = None, destination = None):
        #Origin and destination
        origin, destination = np.random.choice(JUNCTIONS, 2, replace = False)

        self.origin = origin
        self.destination = destination

        #path to follow
        self.path = nx.shortest_path(Graph, self.origin, self.destination, weight = 'length')

        #Velocity
        self.velocity = np.random.choice(self.VELOCITIES, p=self.VELOCITIES_P)

        #Variable to keep track how far into the path we are (to simplify scheduling events)
        #Int between 0 and len(path) - 1 that indicates in which edge we are, starting at 0
        #Essentially, how many edges has it travelled so far
        self.progress = 0

        #Time entrance
        self.time = time_entrance

        #Give it nav with 10% chance
        self.has_nav = np.random.choice([True, False], p=[0.1,0.9])
        self.changed_route = []

        #Schedule next event and store it as attribute
        self.next_event = self.schedule_event_exit()


    def __str__(self):
        at = self.path[self.progress]
        return f"Vehicle travelling from {Graph.nodes[self.origin]['name']} to {Graph.nodes[self.destination]['name']} at {self.velocity} km/h, atm at {Graph.nodes[at]['name']}"



    def custom_weight(self,u,v,data):
        """
        :param u: std for accepting function as weight, node 1
        :param v:  std for accepting function as weight, node 2
        :param data: std for accepting function as weight, edge
        :return: time_to_travel + delay
        """
        length = data['length']
        time_to_travel = self.calc_time_to_travel(length) # Setup mean
        delay = data['accident_duration']
        return time_to_travel + delay


    def calc_time_to_travel(self,length):
        """
        :param length: edge length
        :return: returns time to traverse length based on normal speed
        """
        mean = length / (self.velocity /3.6) #seconds
        std = mean / 20
        time_to_travel = np.random.normal(loc = mean, scale = std) / 3600 #back to hours
        return time_to_travel
    #
    def schedule_event_exit(self):
        if self.progress < len(self.path) - 1:
            # Get the current and next node on the original path
            current_node = self.path[self.progress]
            next_node = self.path[self.progress + 1]
            edge = Graph.edges[(current_node, next_node)]

            # Check for accident on the current edge and if nav is enabled
            if self.has_nav and edge['accident']:
                print("Found an edge with accident")
                new_path = nx.shortest_path(Graph, current_node, self.destination, weight=self.custom_weight)
                if new_path != self.path:
                    print("Found a quicker route")
                    print(f"Prev path = {[Graph.nodes[node]['name'] for node in self.path]}")
                    print(f"New path = {[Graph.nodes[node]['name'] for node in new_path]}")
                    self.changed_route.append((self.path, new_path))

                    # Preserve the already traversed portion
                    prefix = self.path[:self.progress]

                    updated_path = prefix + new_path
                    self.path = updated_path
                    print(f"Updated path: {[Graph.nodes[node]['name'] for node in self.path]}")

                    # Update next_node based on the new path
                    next_node = self.path[self.progress + 1]
                    edge = Graph.edges[(current_node, next_node)]
                    length = edge['length']
                    # Sample travel time along the new edge (including accident delay)
                    time_to_travel = self.calc_time_to_travel(length) + edge['accident_duration']
                else:
                    # No change in path (fallback scenario)
                    length = edge['length']
                    time_to_travel = self.calc_time_to_travel(length)
            else:
                # Normal travel (no accident on the edge)
                length = edge['length']
                time_to_travel = self.calc_time_to_travel(length)

            new_time = self.time + time_to_travel
            # Schedule next event and update car's progress and time
            self.next_event = Event(1, new_time, car=self)
            self.increase_progress()
            self.increase_time(new_time)

            return self.next_event

        # if self.progress == len(self.path) - 1:
        #     print('Car has reached its destination')
        #     self.travel_time = self.next_event.time

    def increase_progress(self):
        self.progress += 1

    def increase_time(self, new_time):
        self.time = new_time

In [66]:
class TowTruck:
    def __init__(self, origin = JUNCTIONS.index('2752332143')):
        self.origin = origin #Gorinchem as specified in assignment 3 except if specified differently
        self.isHome = True

        #Velocity
        self.velocity = 80

        self.progress = 0

        #Time entrance
        # self.time = time_entrance
        #Schedule next event and store it as attribute

        #Placeholder for callback
        self.available_callback = lambda truck: None


        self.next_event = self.schedule_event_exit()
    def __str__(self):
        at = self.path[self.progress]
        return f"Tow Truck travelling from {Graph.nodes[self.origin]['name']} to {Graph.nodes[self.destination]['name']} at {self.velocity} km/h, atm at {Graph.nodes[at]['name']}"

    def calc_time_to_travel(self,length):
        """
        :param length: edge length
        :return: returns time to traverse length based on normal speed
        """
        mean = length / (self.velocity /3.6) #seconds
        std = mean / 20
        time_to_travel = np.random.normal(loc = mean, scale = std) / 3600 #back to hours
        return time_to_travel

    def new_destination(self, destination):
        if self.isHome:
            self.destination = destination
            self.path = nx.shortest_path(Graph, self.origin, self.destination, weight = 'length')
            self.isHome = False
        else:
            print("Not able to set new destination, vehicle still on the road.")
        return

    def schedule_event_exit(self):
        if not self.path:
            return None

        if  self.progress < len(self.path) - 1:
            # Can go on emergency lane so doesn't care for traffic jam
            current_node = self.path[self.progress]
            next_node = self.path[self.progress + 1]

            edge = Graph.edges[(current_node,next_node)]
            length = edge['length']
            time_to_travel = self.calc_time_to_travel(length)

            new_time = self.time + time_to_travel

            #Store event and increase progress
            self.next_event = Event(1 , new_time, car=self)
            self.increase_progress()
            self.increase_time(new_time)

            return self.next_event
        elif not self.isHome and self.progress == len(self.path) - 1:
            # Vehicle has to 'solve accident and drive back'
            #self.next_event = Event(Event.SOLVE_INCIDENT , self.time, car=self, road=self.path[-1])

            # Calculate new route back 'home'
            self.path = nx.shortest_path(Graph, self.destination, self.origin, weight = 'length')
            # Reset progress
            self.progress = 0
            #
            current_node = self.destination
            next_node = self.path[1]

            edge = Graph.edges[(current_node,next_node)]
            length = edge['length']
            time_to_travel = self.calc_time_to_travel(length)
            new_time = self.time + time_to_travel

            self.increase_progress()
            self.increase_time(new_time)
            return self.next_event
        elif self.progress == len(self.path) - 1:
            self.isHome = True
            self.available_callback(self)
            # self.
        # if self.progress == len(self.path) - 1:
        #     print('Car has reached its destination')
        #     self.travel_time = self.next_event.time

    def increase_progress(self):
        self.progress += 1

    def increase_time(self, new_time):
        self.time = new_time


In [ ]:
from collections import deque

class indicent_solvers:
    def __init__(self,K_trucks, origin = JUNCTIONS.index('2752332143')):
        self.origin = origin
        self.K_trucks = K_trucks
        self.trucks = []
        self.available = deque()
        self.unavailable = deque()
        for _ in range(K_trucks):
            truck = TowTruck(0)
            truck.available_callback = self.truck_returned
            self.trucks.append(truck)
            self.available.append(truck)

    def truck_returned(self, truck):
        # Remove the truck from unavailable (if it’s there) and mark it available.
        if truck in self.unavailable:
            self.unavailable.remove(truck)
        self.available.append(truck)
        print(f"Truck returned to base at time {truck.time}")


    def solve_incident(self, destination, incident_duration):
        if len(self.available) > 0:
            truck = self.available.pop()
            # Check if truck can reach in time
            path = nx.shortest_path(Graph, self.origin, destination, weight='length')
            length = nx.path_weight(Graph, path, weight='length')
            if truck.calc_time_to_travel(length) > incident_duration:
                # print("Takes too long to reach incident location")
                self.available.append(truck)
                return
            else:
                # Send the truck to the incident.
                truck.new_destination(destination)
                self.unavailable.append(truck)




In [67]:
#Using a thining approach
max_lambda = np.max(LAMBDA_T) + 1
all_arrivals = arrival_times(max_lambda)

uniform_dist = stats.uniform(0,1)
u_rvs = uniform_dist.rvs(len(all_arrivals))
accept_filter = u_rvs * max_lambda < lambdat(all_arrivals)

accepted_arrivals = all_arrivals[accept_filter]

In [76]:
#Simulation (can be turned into an object later)
LIST_CARS = []
fes = FES()
for arrival in accepted_arrivals:
    #Two events associated with each arrival
    arrival_event = Event(0, arrival)
    fes.add(arrival_event)

Found an edge with accident
Found an edge with accident
Found an edge with accident
Found an edge with accident
Found an edge with accident
Found an edge with accident
Found a quicker route
Prev path = ['Knooppunt Terbregseplein', 'Knooppunt Gouwe', 'Knooppunt Oudenrijn']
New path = ['Knooppunt Terbregseplein', 'Knooppunt Ridderkerk', 'Knooppunt Gorinchem', 'Knooppunt Everdingen', 'Knooppunt Oudenrijn']
Updated path: ['Knooppunt Terbregseplein', 'Knooppunt Terbregseplein', 'Knooppunt Ridderkerk', 'Knooppunt Gorinchem', 'Knooppunt Everdingen', 'Knooppunt Oudenrijn']


KeyError: "The edge (np.str_('8432860337'), np.str_('8432860337')) is not in the graph."

In [70]:
t = 0 #current time
while t < 24.0:
    event = fes.next()
    t = event.time

    if event.type == 0:
        car_travel_event = event.car.next_event
        fes.add(car_travel_event)
        LIST_CARS.append(event.car)

    if event.type == 1:
        next_travel_event_car = event.car.schedule_event_exit()
        if type(next_travel_event_car) == Event: #If the car has arrived to its destination it wont return an event object
            fes.add(next_travel_event_car)
        # else:
            # print(event.car)

    # if event.type == 2: NO ACCIDENTS YET

Found an edge with accident
Found a quicker route
Prev path = ['Knooppunt Gouwe', 'Knooppunt Terbregseplein', 'Knooppunt Ridderkerk', 'Knooppunt Zonzeel', 'Knooppunt Princeville', 'Knooppunt Sint-Annabosch']
new path = ['Knooppunt Terbregseplein', 'Knooppunt Ridderkerk', 'Knooppunt Zonzeel', 'Knooppunt Princeville', 'Knooppunt Sint-Annabosch']
NEW PATH LIST ['Knooppunt Gouwe', 'Knooppunt Terbregseplein', 'Knooppunt Ridderkerk', 'Knooppunt Zonzeel', 'Knooppunt Princeville', 'Knooppunt Sint-Annabosch']


KeyError: "The edge ('8432860337', '43349094') is not in the graph."

In [63]:
#Checking if cars make it to destination
for i in range(0, len(LIST_CARS)):
    car_i = LIST_CARS[i]
    # if car_i.progress != len(car_i.path) - 1:
    #     print(f'oh oh {LIST_CARS[i].time}')
    if car_i.path[car_i.progress] != car_i.destination:
        print(f'oh oh {LIST_CARS[i].time}')
        print(LIST_CARS[i])

oh oh 24.03696884892126
Vehicle travelling from Knooppunt Terbregseplein to Knooppunt Leenderheide at 80 km/h, atm at Knooppunt Batadorp
oh oh 24.00404526316757
Vehicle travelling from Knooppunt Batadorp to Knooppunt Terbregseplein at 80 km/h, atm at Knooppunt Ridderkerk
oh oh 24.11322896306653
Vehicle travelling from Knooppunt Ypenburg to Knooppunt Leenderheide at 100 km/h, atm at Knooppunt Batadorp
oh oh 24.157675078291682
Vehicle travelling from Knooppunt Gouwe to Knooppunt Leenderheide at 100 km/h, atm at Knooppunt Batadorp
oh oh 24.01155397421801
Vehicle travelling from Knooppunt Leenderheide to Knooppunt Gouwe at 100 km/h, atm at Knooppunt Oudenrijn
oh oh 24.02938107639096
Vehicle travelling from Knooppunt Leenderheide to Knooppunt Oudenrijn at 100 km/h, atm at Knooppunt Everdingen
oh oh 24.015063006114012
Vehicle travelling from Knooppunt Terbregseplein to Knooppunt Leenderheide at 100 km/h, atm at Knooppunt De Baars
oh oh 24.036712687897396
Vehicle travelling from Knooppunt De 